This is the first part of a talk at GOSIM 2026, given together with Olivier Grisel (who
does the second half of this talk).
## The Adoption of Python Array API in scikit-learn

### What is the goal of introducing Array API in scikit-learn

- scikit-learn is seasoned a machine learning library that was build with running on
  numpy arrays on CPUs in mind

- with more people using GPUs for ML and AI, more libraries come into play

- since version 1.3 (released in June 2023) we implement support in a growing number or
  functions and estimators in experimental mode

- in the Python ML ecosystem, scikit-learn is an array consuming library

    - a library that gets arrays as inputs and outputs arrays

    - the goal is that during the operations within scikit-learn, the arrays stay on the
      namespace and on the device (specifically GPU devices) they came from for all of
      the computationally intense tasks and for most of the tasks in general
      
    - as a consequence, users in addition to numpy array can pass CuPy arrays and 
    PyTorch tensors; JAX support is in preparation


### Standard

- *Consortium for Python Data API Standards* (consisting of maintainers of array heavy
  Python libraries) developed a few tools that array libraries and array consuming
  libraries such like scikit-learn can use for development

    - [`Array API standard`](https://github.com/data-apis/array-api): defines data
      types, operations and behaviours and publishes a
      [SPEC](https://data-apis.org/array-api/latest/index.html) on it; the spec is versioned and still evolves

    - [`array-api-tests`](https://github.com/data-apis/array-api-tests) suite where
      array libraries can test if they comply to the standard

    - [`array-api-strict`](https://github.com/data-apis/array-api-strict) mimics a
      library that solely emulates the array API standard
        - testing library used to check if we support the minimum standard (even though
          torch and CuPy mostly support additional functionality) on its emulated
          hardware (which makes it runnable easily on the CI)
        - heavily used in scikit-learn CI

    - [`array_api_compat`](https://github.com/data-apis/array-api-compat) bridges gaps
      for parts of numpy, torch and cupy, that don't yet fully support the array API

    - [`array-api-extra`](https://github.com/data-apis/array-api-extra) provides array
      functions not included in the standard but found useful for array-consuming
      libraries

### Passing array API compatible inputs into scikit-learn functions and estimators
- for a full list of supported estimators and functions, see [Array API support
  (experimental)](https://scikit-learn.org/stable/modules/array_api.html)

- since scipy has array API support in experimental mode as well, users need to set 
  `export SCIPY_ARRAY_API=1`

- users currently need to enable array API support by setting a flag on the global
  configuration: </br> `from sklearn import set_config` </br>
  `set_config(array_api_dispatch=True)` </br> or by using the config context manager
  (opt-in)

In [ ]:
# export SCIPY_ARRAY_API=1 # <-- set environment variable

from sklearn.datasets import make_classification
from sklearn import config_context
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import cupy

X_np, y_np = make_classification(random_state=0)
X_cu = cupy.asarray(X_np)
y_cu = cupy.asarray(y_np)
X_cu.device

with config_context(array_api_dispatch=True):
    lda = LinearDiscriminantAnalysis()
    X_trans = lda.fit_transform(X_cu, y_cu)
X_trans.device

- trick: `FunctionTransformer` step that moves feature data from CPU to GPU in Pipelines

### Implementation in scikit-learn

- main tracking issue: [Path for Adopting the Array API
  spec](https://github.com/scikit-learn/scikit-learn/issues/22352)

- scikit-learn is an array consuming library: inputs are arrays + outputs are arrays 

- principle: computations stay on the same device and namespace as the input arrays:

In [3]:
from sklearn.utils._array_api import get_namespace, device

xp, is_array_api = get_namespace(X)  # same library as `X` (CuPy, torch, numpy, strict, …)
device = device(X)                   # same device as `X`
# … 

# inside estimators: 
# keep operations on namespace `xp` and 
# create new arrays on namespace `xp` and `device`


NameError: name 'X' is not defined

- scikit-learn depended on
  [`array_api_compat`](https://github.com/data-apis/array-api-compat) in versions
  1.3-1.6 and now vendors a fixed version since version 1.7
- scikit-learn vendors
  [`array-api-extra`](https://github.com/data-apis/array-api-extra) since version 1.7

- the [`Array API standard`](https://github.com/data-apis/array-api) is a limited subset
  of numpy functionality

  - limited subset of types and operations; for instance no support for sparse arrays
    or matrixes

  - not the whole width of functionality; for instance no u-funcs, sometimes need to
    re-write computational logic for arrays other than numpy

  - some higher order functions are re-created in scikit-learn, mostly by subcasing the
    behavior depending on the different backends and where the function is build from
    lower order function within the array API spec; for instance `_nanmin()`

In [4]:
def _nanmin(X, axis=None, xp=None):
    # TODO: refactor once nan-aware reductions are standardized:
    # https://github.com/data-apis/array-api/issues/621
    xp, _, device_ = get_namespace_and_device(X, xp=xp)
    if _is_numpy_namespace(xp):
        return xp.asarray(numpy.nanmin(X, axis=axis))

    else:
        mask = xp.isnan(X)
        X = xp.min(
            xp.where(mask, xp.asarray(+xp.inf, dtype=X.dtype, device=device_), X),
            axis=axis,
        )
        # Replace Infs from all NaN slices with NaN again
        mask = xp.all(mask, axis=axis)
        if xp.any(mask):
            X = xp.where(mask, xp.asarray(xp.nan, dtype=X.dtype, device=device_), X)
        return X

- since several input arrays can come from different origins/machines and in
  productionalised ML, train and predict often happens on different backends, we
  developed two principles: 
  
  1. `y` follows `X` and `y_true` (`sample_weight`, `labels`, ...)
  follow `y_pred` (or `y_score`)
    - `X` is the larger array where computation speed matters and the cost of moving it
      to another device would be larger
    - `y_pred` is most likely a converted array as an output of a scikit-learn function

  - for the slides: issue [Automatically move y (and sample_weight) to the same device
  and namespace as X](https://github.com/scikit-learn/scikit-learn/issues/28668)

  - support for mixed namespace and device inputs will be added in version 1.9
    - unique approach in array consuming libraries
    - purpose: 1. train models on GPU hardware but deploy predictions on CPU hardware, 2.
      string inputs (only numpy) can be processed next to gpu-arrays

  2. guarantee behaviour when fit and predict should happen on different devices

  - convert estimator attributes to the namespace and device passed as `X` into
    `predict`

    - for the slides: PR [Raise if fit and predict use different array API namespaces or
    devices (continued)](https://github.com/scikit-learn/scikit-learn/pull/33076)

- Cython or C code, computation on sparse arrays for efficiency cannot be bridged

- maintainers often need to compare and benchmark multiple implementation options:
  - trade-offs between runtime speed and memory footprint
  - benchmarking on realistic workloads versus edge cases
  - final choice often involves design discussions between several maintainers

  

### Challenges
- as a consuming library, scikit-learn depends on upstream implementations, from numpy
  and scipy; especially scipy is blocking us sometimes

- plans of moving array API support in scikit-learn out of experimental mode are in the
  making

  - specifically we need to wait for scipy to move out of experimental, since sklearn
    should not turn on SciPy's experimental, process-wide mode on the user's behalf and
    a non-experimental mode in scikit-learn needs a stable non-surprising usage of
    different array types in scipy
    
  - for the slides: issue [RFC a plan to move array API support out of
  experimental](https://github.com/scikit-learn/scikit-learn/issues/33444)

- scikit-learn's compiled code written in Cython, C or C++ is optimised to run on CPUs

  - integrating these with the Python Array API is an ongoing challenge

  - estimators that do the heavy part of the computation on compiled code can either 

    - a) convert (copy) the relevant arrays to numpy thus transferring them to CPU, pass
      them into the compiled computation and then converting them back to the input
      namespace and device

    - b) keep the compiled version only for the numpy case and do GPU optimised array
      transformations in Python instead
      
  - discovering if plugins can be used as an alternative


### Sources

- Array API SPEC: [Python array API
  standard](https://data-apis.org/array-api/latest/index.html), Consortium for Python
  Data API Standards.

- scikit-learn's documentation on Array API implementation: [Array API support
  (experimental)](https://scikit-learn.org/dev/modules/array_api.html).

- issue [Path for Adopting the Array API
  spec](https://github.com/scikit-learn/scikit-learn/issues/22352)

- issue [RFC a plan to move array API support out of
  experimental](https://github.com/scikit-learn/scikit-learn/issues/33444)

### References

- Lucy Liu: [Update on array API adoption in
  scikit-learn](https://labs.quansight.org/blog/array-api-scikit-learn-2026), Quansight
  Labs Blog, March 6 2026.
  
- Thomas J. Fan: [Array API Support in
  scikit-learn](https://labs.quansight.org/blog/array-api-support-scikit-learn),
  Quansight Labs Blog, September 19 2023.